# Deep Learning Lab: compact, swappable template

Set `Q` and run the last cell. Questions from the DOCX are grouped below; the same
small function is reused for each question in a group.

| Q | Template |
|---|---|
| 1-5 | XOR / perceptron / hidden-size comparison |
| 6-12 | MNIST ANN / hidden-size / dropout |
| 13-18 | basic or convolutional autoencoder / denoising / anomaly score |
| 19-24 | MFCC speech-command classifier (supply `speech_dir`) |
| 25 | Whisper speech-to-text (supply `audio_file`) |
| 26, 32 | image classifier (CIFAR-10 demo; replace with vehicle/sign folders) |
| 27-31 | YOLO detection, tracking, counting, density, helmet/sign adaptation |
| 33, 38 | tabular autoencoder anomaly detection |
| 34, 36, 37 | fraud ANN / class weights / Random Forest |
| 35 | chronological LSTM forecast and error flags |
| 39-42 | RBM reconstruction, generation, features |
| 43-48 | IMDB LSTM sentiment (replace text/labels for tweets/feedback) |

Install only what the selected question needs, for example:
`pip install numpy scikit-learn tensorflow pandas librosa soundfile openai-whisper ultralytics`.


In [ ]:
# Type words from the experiment title; the closest template is selected.
TITLE = "smart door unlocks when exactly one sensor is active"
EPOCHS = 2
speech_dir = "data/speech"       # class folders for Q19-24
audio_file = "audio.wav"         # Q25
table_file = "data/data.csv"     # Q33-38
target = "Class"                 # target column for Q34-37


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

def show(y, p):
    print("accuracy:", round(accuracy_score(y, p), 4))
    print(np.c_[y[:10], p[:10]])

def xor(q=1):
    from sklearn.linear_model import Perceptron
    from sklearn.neural_network import MLPClassifier
    X = np.array([[0,0],[0,1],[1,0],[1,1]])
    y = np.array([0,1,1,0])
    if q == 4:
        p = Perceptron(max_iter=2000, random_state=1).fit(X, y).predict(X)
        print("perceptron"); show(y, p)
    sizes = [1,2,4,8] if q == 5 else [4]
    for h in sizes:
        m = MLPClassifier((h,), activation="tanh", solver="lbfgs",
                          max_iter=3000, random_state=1).fit(X, y)
        print("hidden:", h); show(y, m.predict(X))


In [ ]:
def mnist_ann(q=6):
    from tensorflow import keras
    from tensorflow.keras import layers
    (x,y),(xt,yt) = keras.datasets.mnist.load_data()
    x,xt = x[...,None]/255., xt[...,None]/255.
    hs = [64,128,256] if q == 10 else [128]
    for h in hs:
        m = keras.Sequential([layers.Input((28,28,1)), layers.Flatten(),
            layers.Dense(h,"relu"), layers.Dropout(.3 if q == 12 else 0),
            layers.Dense(10,"softmax")])
        m.compile("adam","sparse_categorical_crossentropy",metrics=["accuracy"])
        m.fit(x,y,epochs=EPOCHS,batch_size=256,verbose=0)
        print("hidden:",h,"test:",m.evaluate(xt,yt,verbose=0)[1])
    return m

def autoencoder(q=13):
    from tensorflow import keras
    from tensorflow.keras import layers
    (x,_),(xt,_) = keras.datasets.mnist.load_data()
    x,xt = x[...,None]/255., xt[...,None]/255.
    noisy = np.clip(x + .2*np.random.randn(*x.shape),0,1)
    inp=keras.Input((28,28,1))
    if q == 16:
        z=layers.Conv2D(8,3,2,"same",activation="relu")(inp)
        out=layers.Conv2DTranspose(1,3,2,"same",activation="sigmoid")(z)
    else:
        z=layers.Flatten()(inp); z=layers.Dense(32,"relu")(z)
        out=layers.Dense(784,"sigmoid")(z); out=layers.Reshape((28,28,1))(out)
    m=keras.Model(inp,out)
    m.compile("adam","mse")
    m.fit(noisy if q==14 else x, x, epochs=EPOCHS,batch_size=256,verbose=0)
    r=m.predict(xt,verbose=0); err=np.mean((xt-r)**2,(1,2,3))
    print("mean reconstruction error:",err.mean(),"top indexes:",err.argsort()[-10:])
    return m


In [ ]:
def speech(q=19, root=speech_dir):
    import librosa, pathlib
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.neural_network import MLPClassifier
    X=[]; y=[]
    for f in pathlib.Path(root).rglob("*.wav"):
        a,s=librosa.load(f,sr=16000); X.append(librosa.feature.mfcc(y=a,sr=s,n_mfcc=20).mean(1)); y.append(f.parent.name)
    if not X: raise FileNotFoundError("Put labeled WAV files in class folders under speech_dir.")
    yy=LabelEncoder().fit_transform(y); a,b,c,d=train_test_split(X,yy,test_size=.2,stratify=yy,random_state=1)
    m=MLPClassifier((64,),max_iter=300,random_state=1).fit(a,c)
    print("MFCC speech accuracy:",m.score(b,d)); return m

def transcribe(path=audio_file):
    import whisper
    text=whisper.load_model("base").transcribe(path)["text"]
    print(" ".join(text.split())); return text

def images(q=26):
    from tensorflow import keras
    from tensorflow.keras import layers
    (x,y),(xt,yt)=keras.datasets.cifar10.load_data()
    m=keras.Sequential([layers.Input((32,32,3)),layers.Rescaling(1/255),
        layers.Conv2D(16,3,"relu"),layers.MaxPooling2D(),layers.Flatten(),
        layers.Dense(10,"softmax")])
    m.compile("adam","sparse_categorical_crossentropy",metrics=["accuracy"])
    m.fit(x,y,epochs=EPOCHS,batch_size=256,verbose=0)
    print("image accuracy:",m.evaluate(xt,yt,verbose=0)[1])
    print("For Q27-31 use: from ultralytics import YOLO; YOLO('yolo11n.pt').track(source='video.mp4',show=True)")


In [ ]:
def tabular(q=33, path=table_file):
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report
    df=pd.read_csv(path).dropna(); y=df.pop(target).to_numpy() if target in df else None
    X=pd.get_dummies(df).to_numpy("float32"); X=StandardScaler().fit_transform(X)
    if q == 35:
        from tensorflow import keras
        from tensorflow.keras import layers
        v=df.iloc[:, 0].to_numpy("float32")
        n=10; a=np.array([v[i:i+n] for i in range(len(v)-n)])
        b=v[n:]; cut=int(len(a)*.8)
        m=keras.Sequential([layers.Input((n,1)),layers.LSTM(16),layers.Dense(1)])
        m.compile("adam","mse"); m.fit(a[:cut,...,None],b[:cut],epochs=EPOCHS,verbose=0)
        e=np.abs(b[cut:]-m.predict(a[cut:,...,None],verbose=0).ravel())
        print("forecast error threshold:",np.quantile(e,.99)); return m
    if q in (34,36,37) and y is not None:
        a,b,c,d=train_test_split(X,y,test_size=.2,stratify=y,random_state=1)
        if q == 37:
            m=RandomForestClassifier(100,class_weight="balanced",random_state=1)
        else:
            from sklearn.neural_network import MLPClassifier
            m=MLPClassifier((32,),max_iter=200,random_state=1)
            if q == 36:
                z=np.bincount(c); r=np.random.default_rng(1)
                ix=np.concatenate([r.choice(np.where(c==k)[0],z.max(),True)
                    for k in range(len(z))])
                a,c=a[ix],c[ix]
        m.fit(a,c); print(classification_report(d,m.predict(b),zero_division=0)); return m
    from sklearn.neural_network import MLPRegressor
    a,b=train_test_split(X,test_size=.2,random_state=1); m=MLPRegressor((16,),max_iter=300,random_state=1).fit(a,a)
    e=((b-m.predict(b))**2).mean(1); print("threshold:",np.quantile(e,.99),"anomalies:",sum(e>np.quantile(e,.99)))

def sentiment(q=43):
    from tensorflow import keras
    from tensorflow.keras import layers
    (x,y),(xt,yt)=keras.datasets.imdb.load_data(num_words=10000)
    x=keras.utils.pad_sequences(x,maxlen=200); xt=keras.utils.pad_sequences(xt,maxlen=200)
    m=keras.Sequential([layers.Embedding(10000,32),layers.LSTM(32),layers.Dense(1,"sigmoid")])
    m.compile("adam","binary_crossentropy",metrics=["accuracy"]); m.fit(x,y,epochs=EPOCHS,batch_size=256,verbose=0)
    print("sentiment accuracy:",m.evaluate(xt,yt,verbose=0)[1]); return m


In [ ]:
# Q39-42: compact RBM demonstration.
def rbm():
    from sklearn.neural_network import BernoulliRBM
    from tensorflow import keras
    (x,_),_=keras.datasets.mnist.load_data(); x=(x[:2000].reshape(-1,784)>127).astype("float32")
    m=BernoulliRBM(32,n_iter=5,random_state=1).fit(x)
    z=m.transform(x); r=m.inverse_transform(z)
    print("RBM reconstruction MSE:",((x-r)**2).mean(),"features:",z.shape); return m

def pick(s):
    s=s.lower()
    groups=[(("xor","sensor","alarm","door"),1),(("digit","character","handwritten","dropout"),6),
      (("xray","reconstruct","denois","autoencoder"),13),(("speech","mfcc","spoken","keyword"),19),
      (("transcri","lecture","whisper"),25),(("vehicle","traffic sign","image classify"),26),
      (("yolo","track","count vehicle","helmet"),27),(("fraud","transaction","stock volume"),33),
      (("forecast","stock price","time series"),35),(("rbm","synthetic digit"),39),
      (("sentiment","review","tweet","complaint","feedback"),43)]
    return max(groups,key=lambda g:sum(w in s for w in g[0]))[1]
T=pick(TITLE); print("matched template group:",T)
if T == 1: result=xor(1)
elif T == 6: result=mnist_ann(6)
elif T == 13: result=autoencoder(13)
elif T == 19: result=speech(19)
elif T == 25: result=transcribe()
elif T in (26,27): result=images(T)
elif T == 33: result=tabular(33)
elif T == 35: result=tabular(35)
elif T == 39: result=rbm()
else: result=sentiment(43)
